In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

emb = sc.read_h5ad("/mnt/172/wh/25-12/spatial/adata_with_embeddings.h5ad")
emb

AnnData object with n_obs × n_vars = 368226 × 20310
    obs: 'assay', 'organism', 'nicheformer_split', 'batch', 'niche', 'region', 'author_cell_type', 'modality', 'specie'
    uns: 'technology_mean'
    obsm: 'X_niche_0', 'X_niche_1', 'X_niche_2', 'X_niche_3', 'X_niche_4', 'X_nicheformer_embeddings', 'X_pca', 'X_umap', 'X_umap_bySlide', 'falsecode_counts', 'negprobes_counts', 'spatial', 'spatial_fov_px'
    layers: 'counts', 'data'

In [2]:
# The embeddings: 512-dim vector per cell
emb.obsm['X_nicheformer_embeddings']

array([[-0.4986512 , -0.45095646,  0.19919908, ..., -0.19404154,
        -0.22005968,  0.1734344 ],
       [-0.56532323, -0.38802993,  0.09161816, ..., -0.12084281,
        -0.17405882, -0.08705514],
       [-0.5355022 , -0.28048816,  0.12545146, ..., -0.15409243,
        -0.2330774 ,  0.03191693],
       ...,
       [-0.61504006, -0.149122  , -0.05951155, ..., -0.04556726,
        -0.08118979,  0.12945957],
       [-0.7225345 , -0.08533221,  0.11955155, ..., -0.17251487,
        -0.13314742,  0.28851673],
       [-0.4970719 , -0.17842384,  0.01142836, ..., -0.18797292,
        -0.12397236,  0.00359809]], dtype=float32)

---
## Downstream Usage of Nicheformer Embeddings

The `X_nicheformer_embeddings` key contains a 512-dimensional vector for each of the 368,226 cells. Below are practical downstream analyses.

### 1. UMAP Visualization colored by cell type / region / niche

In [3]:
# Compute UMAP on the Nicheformer embeddings directly
import scanpy as sc

sc.pp.neighbors(emb, use_rep='X_nicheformer_embeddings')
sc.tl.umap(emb, min_dist=0.3)

# Store in a dedicated key
emb.obsm['X_emb_umap'] = emb.obsm['X_umap'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(emb, color='author_cell_type', ax=axes[0], show=False, title='Cell Type')
sc.pl.umap(emb, color='region', ax=axes[1], show=False, title='Region')
sc.pl.umap(emb, color='niche', ax=axes[2], show=False, title='Niche')
plt.tight_layout()
plt.show()

2026-05-11 11:22:57.575512: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 11:22:57.597585: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-11 11:22:57.624572: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-11 11:22:57.633209: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-11 11:22:57.650637: I tensorflow/core/platform/cpu_feature_guar

NameError: name 'plt' is not defined

### 2. Clustering on Embeddings

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Cluster embeddings into 15 groups
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
emb.obs['emb_cluster'] = kmeans.fit_predict(emb.obsm['X_nicheformer_embeddings']).astype(str)

# Cross-tabulation with known cell types
ctab = pd.crosstab(emb.obs['emb_cluster'], emb.obs['author_cell_type'])
print("Cluster × Cell Type contingency table (top-left 10×6):")
display(ctab.iloc[:10, :6])

# ARI vs ground truth
ari = adjusted_rand_score(emb.obs['emb_cluster'], emb.obs['author_cell_type'])
print(f"\nAdjusted Rand Index vs author_cell_type: {ari:.3f}")

### 3. Cell-Type Classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = emb.obsm['X_nicheformer_embeddings']
y = emb.obs['author_cell_type'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train a Random Forest
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### 4. Niche Composition Regression

Predict the spatial niche composition vectors (`X_niche_0`..`X_niche_4`) from the embeddings.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

X = emb.obsm['X_nicheformer_embeddings']

for niche_key in ['X_niche_0', 'X_niche_1', 'X_niche_2', 'X_niche_3', 'X_niche_4']:
    y = emb.obsm[niche_key]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{niche_key:12s} → R²={r2:.3f}, RMSE={rmse:.4f}")

### 5. Batch Correction with Harmony

Use the embeddings as input to Harmony for batch integration across slides/assays.

In [ ]:
try:
    import harmonypy
    
    # Harmony integration on the embeddings
    sc.external.pp.harmony_integrate(
        emb, key='batch',
        basis='X_nicheformer_embeddings',
        adjusted_basis='X_emb_harmony'
    )
    
    # Compute UMAP on harmonized embeddings
    sc.pp.neighbors(emb, use_rep='X_emb_harmony')
    sc.tl.umap(emb)
    emb.obsm['X_emb_harmony_umap'] = emb.obsm['X_umap'].copy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sc.pl.umap(emb, color='batch', ax=axes[0], show=False, title='Harmony: Batch')
    sc.pl.umap(emb, color='author_cell_type', ax=axes[1], show=False, title='Harmony: Cell Type')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("harmonypy not installed. Install with: pip install harmonypy")

### 6. Gene Expression Imputation from Embeddings

Predict expression of specific genes from the Nicheformer embeddings.

In [ ]:
from sklearn.linear_model import Ridge
from scipy.sparse import issparse

# Pick a few genes to test imputation
gene_names = ['CD3E', 'CD8A', 'CD4', 'EPCAM', 'VIM', 'COL1A1']
available_genes = [g for g in gene_names if g in emb.var_names]
print(f"Testing imputation for: {available_genes}")

X = emb.obsm['X_nicheformer_embeddings']

for gene in available_genes:
    gene_idx = list(emb.var_names).index(gene)
    y = emb[:, gene_idx].X
    if issparse(y):
        y = y.toarray().flatten()
    else:
        y = y.flatten()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    print(f"  {gene:8s} → R²={r2:.3f}")

### 7. Spatial Visualization

If spatial coordinates are available, visualize embeddings in physical space.

In [ ]:
# Check if spatial coordinates exist
if 'spatial' in emb.obsm:
    spatial_coords = emb.obsm['spatial']
    print(f"Spatial coordinates shape: {spatial_coords.shape}")
    
    # Subsample for plotting (large dataset)
    n_sample = min(50000, emb.n_obs)
    idx_sample = np.random.choice(emb.n_obs, n_sample, replace=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot first 2 dims of embedding as spatial overlay
    sc = axes[0].scatter(
        spatial_coords[idx_sample, 0],
        spatial_coords[idx_sample, 1],
        c=emb.obsm['X_nicheformer_embeddings'][idx_sample, 0],
        s=1, cmap='viridis', alpha=0.6
    )
    axes[0].set_title('Embedding dim-0 in spatial space')
    plt.colorbar(sc, ax=axes[0])
    
    # Plot cell type in spatial space
    cell_types = emb.obs['author_cell_type'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(cell_types)))
    ct_to_color = dict(zip(cell_types, colors))
    
    for ct in cell_types:
        mask = emb.obs['author_cell_type'].values[idx_sample] == ct
        axes[1].scatter(
            spatial_coords[idx_sample[mask], 0],
            spatial_coords[idx_sample[mask], 1],
            c=[ct_to_color[ct]], s=1, alpha=0.6, label=ct
        )
    axes[1].set_title('Cell Types in Spatial Space')
    axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6)
    
    plt.tight_layout()
    plt.show()
else:
    print("No spatial coordinates found in emb.obsm['spatial']")

### 8. Export Embeddings for External Use

Save the embeddings as NumPy arrays or CSV for use in other tools.

In [ ]:
import os

output_dir = "/mnt/172/wh/25-12/spatial/output"
os.makedirs(output_dir, exist_ok=True)

# Save embeddings as .npy
np.save(os.path.join(output_dir, "nicheformer_embeddings.npy"),
        emb.obsm['X_nicheformer_embeddings'])

# Save metadata as CSV
metadata_cols = ['assay', 'organism', 'nicheformer_split', 'batch',
                 'niche', 'region', 'author_cell_type', 'modality', 'specie']
metadata_df = emb.obs[metadata_cols].copy()
metadata_df.to_csv(os.path.join(output_dir, "nicheformer_metadata.csv"))

print(f"Embeddings saved to: {output_dir}/nicheformer_embeddings.npy")
print(f"  Shape: {emb.obsm['X_nicheformer_embeddings'].shape}")
print(f"Metadata saved to: {output_dir}/nicheformer_metadata.csv")
print(f"  Shape: {metadata_df.shape}")

---
## Summary

| Analysis | Key | Description |
|----------|-----|-------------|
| UMAP | `emb.obsm['X_emb_umap']` | 2D visualization of embeddings |
| Clustering | `emb.obs['emb_cluster']` | KMeans clusters (15 groups) |
| Classification | — | RandomForest → cell types |
| Niche Regression | — | Ridge → predict `X_niche_*` |
| Batch Correction | `emb.obsm['X_emb_harmony']` | Harmony-integrated embeddings |
| Gene Imputation | — | Ridge → predict gene expression |
| Spatial Overlay | — | Embedding dims in physical space |
| Export | `.npy` / `.csv` | For external tools |